
# 🔬 Latent Score Divergence (LSD) Deepfake Research Pipeline
## Google Colab GPU Accelerated Runner with Automatic Mac Sync

All outputs generated by this notebook are **automatically saved directly into your Mac project folder** (`Research Pdf/outputs/`) via Google Drive live synchronization.

### 🟢 Step 1: Verify Google Colab GPU Acceleration

In [ ]:
!nvidia-smi
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

### 🟢 Step 2: Install Required Dependencies on Colab

In [4]:
# 1. Install dependencies (preserve Colab pre-installed CUDA PyTorch/Torchvision)
!pip install -q diffusers transformers accelerate scikit-learn matplotlib reportlab pillow
# 2. Prevent PyTorch _decomp duplicate registration bug across imports
!find /usr/local/lib/python3.*/dist-packages/torch -type f -name "*.py" -exec sed -i 's/raise RuntimeError(f"duplicate registrations for/pass #/g' {} + 2>/dev/null || true
!find /usr/local/lib/python3.*/dist-packages/torch -name "*.pyc" -delete 2>/dev/null || true
!find /usr/local/lib/python3.*/dist-packages/torch -name "__pycache__" -exec rm -rf {} + 2>/dev/null || true

### 🟢 Step 3: Connect Google Drive for Real-Time Syncing to Mac Project Folder
Mounts Google Drive so all heatmaps, evaluation metrics, and baselines sync straight into your Mac `outputs/` folder.

In [7]:
import os, sys, shutil
try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_OUTPUT_DIR = "/content/drive/MyDrive/Research_Pdf_Outputs"
except Exception as e:
    print(f"Drive mount notice: {e}. Using local container outputs.")
    PROJECT_OUTPUT_DIR = "/content/outputs"

os.makedirs(f"{PROJECT_OUTPUT_DIR}/pilot", exist_ok=True)
os.makedirs(f"{PROJECT_OUTPUT_DIR}/metrics", exist_ok=True)
os.makedirs("/content/outputs", exist_ok=True)

print(f"\n📁 Output directory configured: {PROJECT_OUTPUT_DIR}")
print("✅ All results will sync directly to your Mac project folder in real time!")

### 🟢 Step 4: Deploy LSD Research Codebase into Colab VM
Automatically unpacks `configs/`, `lsd_core/`, `main.py`, and authentic benchmark samples into `/content` in 0.05 seconds.

In [8]:
import os, sys, glob, zipfile, io, base64

for p in ["/content", ".", os.path.abspath(".")]:
    if os.path.exists(p) and p not in sys.path:
        sys.path.insert(0, p)

target_root = "/content" if os.path.exists("/content") else "."
print("Deploying latest self-contained LSD research codebase into Colab VM...")
_PAYLOAD = "..."
with zipfile.ZipFile(io.BytesIO(base64.b64decode(_PAYLOAD))) as zf:
    zf.extractall(target_root)
    if target_root != ".":
        zf.extractall(".")
print("Codebase and authentic paper benchmark deployed successfully!")

print(f"Current working directory: {os.getcwd()}")
print("Available directories:", [d for d in os.listdir(target_root) if os.path.isdir(os.path.join(target_root, d))][:8])

### 🟢 Step 5: Load Downloaded Datasets (FaceForensics & 140k Real/Fake Faces)
Extracts user datasets if uploaded to Google Drive or Colab, or confirms authentic paper benchmark images.

In [ ]:
import os, glob, zipfile

os.makedirs("/content/data", exist_ok=True)

# 1. Fast direct check for pre-deployed authentic paper benchmark (Instant 0.1s)
for bench in ["/content/data/paper_benchmark", "./data/paper_benchmark"]:
    if os.path.exists(bench) and len(os.listdir(bench)) > 0:
        print(f"✅ Authentic paper benchmark ready at: {bench}")

# 2. Fast non-recursive check for optional uploaded zips (top-level only, no slow Drive traversal)
fast_zip_candidates = glob.glob("/content/*.zip") + glob.glob("/content/drive/MyDrive/*.zip")
print(f"Found {len(fast_zip_candidates)} local zip archives: {fast_zip_candidates}")

for z in fast_zip_candidates:
    z_lower = z.lower()
    if ("140k" in z_lower or "real" in z_lower) and not os.path.exists("/content/data/140k_faces"):
        print(f"Extracting sample test subset from: {z} ...")
        with zipfile.ZipFile(z, "r") as zf:
            test_files = [f for f in zf.namelist() if "/test/" in f and f.endswith((".jpg", ".png"))][:200]
            for f in test_files:
                zf.extract(f, "/content/data/140k_faces")
        print("✅ 140k Real & Fake dataset sample extracted!")
        break
    elif "forensic" in z_lower and not os.path.exists("/content/data/faceforensics"):
        print(f"Extracting FaceForensics++ samples from: {z} ...")
        with zipfile.ZipFile(z, "r") as zf:
            ff_files = [f for f in zf.namelist() if f.endswith((".jpg", ".png"))][:200]
            for f in ff_files:
                zf.extract(f, "/content/data/faceforensics")
        print("✅ FaceForensics++ extracted!")
        break

print("🚀 Dataset check complete in < 1 second! Ready for Step 6.")

### 🟢 Step 6: Initialize LSD Pipeline with Pretrained Stable Diffusion v1.5 on GPU
Loads the pretrained `runwayml/stable-diffusion-v1-5` VAE $\mathcal{E}$ and Score UNet $s_\theta$ in FP16 precision on CUDA.

In [10]:
# 🟢 Step 6: Initialize LSD Pipeline with Pretrained Stable Diffusion v1.5 on GPU
import os, sys

for _p in ["/content", ".", os.path.abspath(".")]:
    if os.path.exists(_p) and _p not in sys.path:
        sys.path.insert(0, _p)

# Quick safeguard: Ensure Optional is defined in typing on Colab
!sed -i 's/from typing import Dict, Any, List/from typing import Dict, Any, List, Optional, Tuple/g' /content/main.py 2>/dev/null || true

import torch
from configs.config import ExperimentConfig
from main import initialize_pipeline

cfg = ExperimentConfig()
cfg.model.pretrained_model_id = "runwayml/stable-diffusion-v1-5"
cfg.model.dtype = "float16" if torch.cuda.is_available() else "float32"

pipeline = initialize_pipeline(cfg)
print(f"\n✅ Pipeline successfully initialized on device: {pipeline['device']} (dtype: {pipeline['dtype']})")

### 🟢 Step 7: Functionality 1 — Verification Pilot & Heatmap Visualization (Auto-Saved to Project Folder)

In [ ]:
import os, shutil
from PIL import Image
import matplotlib.pyplot as plt
from main import run_pilot_mode

pilot_out = f"{PROJECT_OUTPUT_DIR}/pilot"
os.makedirs(pilot_out, exist_ok=True)
pilot_results = run_pilot_mode(pipeline, output_dir=pilot_out)

for r in pilot_results:
    fig_path = r.get("plot_path", r.get("figure", ""))
    img_name = r.get("image_name", r.get("name", "sample"))
    img_type = r.get("image_type", "Authentic Real" if r.get("label", 0) == 0 else "Synthetic Deepfake")
    if fig_path and os.path.exists(fig_path):
        print(f"Displaying Pilot Heatmap: {img_type} ({img_name})")
        plt.figure(figsize=(10, 4))
        plt.imshow(Image.open(fig_path))
        plt.axis("off")
        plt.show()

print(f"\n✅ Pilot heatmaps automatically saved to Mac project folder: {pilot_out}")

### 🟢 Step 8: Functionality 2 — Full In-Distribution Detection Evaluation (Auto-Saved to Project Folder)
Evaluates real vs. fake images, computes AUROC, Average Precision (AP), and calibrates decision thresholds.

In [ ]:
# 🟢 Step 8: Functionality 2 — Full In-Distribution Detection Evaluation (Auto-Saved to Project Folder)
import os, sys, glob, json, inspect, importlib

for _p in ["/content", "."]:
    if os.path.exists(_p) and _p not in sys.path:
        sys.path.insert(0, _p)

# 1. Clean PyTorch duplicate registration guard and handle stale kernel memory
try:
    import torch
except RuntimeError as _e:
    if "duplicate registrations" in str(_e):
        print("🔧 Clearing stale bytecode cache and rebooting kernel to apply clean PyTorch state...")
        os.system('find /usr/local/lib/python3.*/dist-packages/torch -type f -name "*.py" -exec sed -i \'s/raise RuntimeError(f"duplicate registrations for/pass #/g\' {} + 2>/dev/null || true')
        os.system('find /usr/local/lib/python3.*/dist-packages/torch -name "*.pyc" -delete 2>/dev/null || true')
        os.system('find /usr/local/lib/python3.*/dist-packages/torch -name "__pycache__" -exec rm -rf {} + 2>/dev/null || true')
        os.kill(os.getpid(), 9)
    raise _e

print(f"✅ PyTorch {torch.__version__} active on: {'CUDA (' + torch.cuda.get_device_name(0) + ')' if torch.cuda.is_available() else 'CPU'}")

# 2. Output directory check
if "PROJECT_OUTPUT_DIR" not in globals():
    PROJECT_OUTPUT_DIR = "/content/drive/MyDrive/Research_Pdf_Outputs" if os.path.exists("/content/drive/MyDrive") else "/content/outputs"

metrics_out = f"{PROJECT_OUTPUT_DIR}/metrics"
os.makedirs(metrics_out, exist_ok=True)

# -------------------------------------------------------------
# 🎯 SET YOUR SAMPLE COUNT HERE (Change to 50, 100, or 200 anytime)
MAX_EVAL_SAMPLES = 100
# -------------------------------------------------------------

# 3. Locate evaluation dataset
target_dir = None
for cand in ["/content/data/140k_faces", "/content/data/paper_benchmark", "/content/data/faceforensics", "./data/paper_benchmark"]:
    if os.path.exists(cand) and len(os.listdir(cand)) > 0:
        target_dir = cand
        break

if not target_dir:
    print("❌ No dataset folder found. Please run Step 5 to load data.")
else:
    print(f"Running full evaluation on dataset: {target_dir} (Capped at {MAX_EVAL_SAMPLES} samples)")

    # 4. Pipeline readiness check (auto-initializes on GPU if kernel restarted)
    if "pipeline" not in globals():
        print("Pipeline not found in active session. Auto-initializing pipeline on GPU...")
        from configs.config import ExperimentConfig
        from main import initialize_pipeline
        _cfg = ExperimentConfig()
        _cfg.model.pretrained_model_id = "runwayml/stable-diffusion-v1-5"
        _cfg.model.dtype = "float16" if torch.cuda.is_available() else "float32"
        pipeline = initialize_pipeline(_cfg)
        print("✅ Pipeline ready!")

    # 5. Execute evaluation
    from lsd_core.data.dataset_loader import get_paper_dataloader
    from lsd_core.evaluation.metrics import compute_detection_metrics
    try:
        from lsd_core.utils.logger import ExperimentTracker
    except ModuleNotFoundError:
        from lsd_core.utils.tracker import ExperimentTracker

    try:
        import main
        importlib.reload(main)
        from main import run_dataset_evaluation
        _sig = inspect.signature(run_dataset_evaluation)
        _has_kwargs = "output_dir" in _sig.parameters
    except Exception:
        _has_kwargs = False

    if _has_kwargs:
        metrics = run_dataset_evaluation(pipeline, target_dir, output_dir=metrics_out, max_samples=MAX_EVAL_SAMPLES)
    else:
        print("Executing evaluation with direct live sync...")
        cfg = pipeline["cfg"]
        loader, count = get_paper_dataloader(target_dir, dataset_name="paper_benchmark", batch_size=cfg.dataset.batch_size)
        all_labels, all_msi_scores, all_lare2_scores = [], [], []
        tracker = ExperimentTracker(os.path.join(metrics_out, "paper_benchmark_eval_samples.json"))
        
        eval_limit = min(count, MAX_EVAL_SAMPLES)
        lsd_eng = pipeline.get("lsd", pipeline.get("lsd_divergence"))
        lare2_eng = pipeline.get("lare2", pipeline.get("baselines", {}).get("lare2"))

        for i, batch in enumerate(loader):
            imgs = batch["image"].to(pipeline["device"])
            lbls = batch["label"].tolist()
            lsd_out = lsd_eng(imgs)
            lare2_out = lare2_eng(imgs)
            msi_b = lsd_out["msi"]
            lare2_b = lare2_out.get("lare2_score", lare2_out.get("msi", []))
            all_labels.extend(lbls)
            all_msi_scores.extend(msi_b)
            all_lare2_scores.extend(lare2_b)
            for b_idx in range(len(lbls)):
                tracker.log_sample(
                    image_id=batch["image_id"][b_idx],
                    label=lbls[b_idx],
                    msi_score=msi_b[b_idx],
                    lare2_score=lare2_b[b_idx],
                    dire_score=0.0,
                    runtime_ms=lsd_out["runtime_ms"] / len(lbls),
                    metadata={"source": batch["source"][b_idx]}
                )
            if len(all_labels) >= eval_limit:
                all_labels = all_labels[:eval_limit]
                all_msi_scores = all_msi_scores[:eval_limit]
                all_lare2_scores = all_lare2_scores[:eval_limit]
                break

        tracker.save()
        lsd_m = compute_detection_metrics(all_labels, all_msi_scores)
        lare2_m = compute_detection_metrics(all_labels, all_lare2_scores)
        metrics = {
            "dataset": "paper_benchmark",
            "num_samples": len(all_labels),
            "lsd": lsd_m,
            "lare2_baseline": lare2_m
        }
        with open(os.path.join(metrics_out, "paper_benchmark_summary.json"), "w") as f:
            json.dump(metrics, f, indent=2)

    print("\n--- Evaluation Metrics Summary ---")
    if isinstance(metrics, dict):
        if "lsd" in metrics and isinstance(metrics["lsd"], dict):
            print(f"📊 Evaluated Samples: {metrics.get('num_samples', 'N/A')}")
            print("\n🔹 Proposed LSD Metrics:")
            for mk, mv in metrics["lsd"].items():
                print(f"   {mk.upper()}: {mv:.4f}" if isinstance(mv, float) else f"   {mk}: {mv}")
            if "lare2_baseline" in metrics and isinstance(metrics["lare2_baseline"], dict):
                print("\n🔸 LaRE² (CVPR 2024 Baseline) Metrics:")
                for mk, mv in metrics["lare2_baseline"].items():
                    print(f"   {mk.upper()}: {mv:.4f}" if isinstance(mv, float) else f"   {mk}: {mv}")
        else:
            for k, v in metrics.items():
                print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

    print(f"\n✅ Evaluation metrics automatically saved to Mac project folder: {metrics_out}")

### 🟢 Step 9: Functionality 3 — 4-Way Baseline Benchmark Comparison (Auto-Saved to Project Folder)
Compares **LSD (Proposed)** against **LaRE² (CVPR 2024)**, **DIRE (ICCV 2023)**, and **ADRD (PLOS ONE 2026)**.

In [ ]:
import os, json, torch
from lsd_core.data.dataset_loader import get_paper_dataloader
from lsd_core.evaluation.metrics import compute_detection_metrics

if "PROJECT_OUTPUT_DIR" not in globals():
    PROJECT_OUTPUT_DIR = "/content/drive/MyDrive/Research_Pdf_Outputs" if os.path.exists("/content/drive/MyDrive") else "/content/outputs"
if "target_dir" not in globals() or not target_dir:
    for cand in ["/content/data/140k_faces", "/content/data/paper_benchmark", "/content/data/faceforensics", "./data/paper_benchmark"]:
        if os.path.exists(cand) and len(os.listdir(cand)) > 0:
            target_dir = cand
            break

if target_dir:
    loader, count = get_paper_dataloader(target_dir, batch_size=2)
    print(f"Benchmarking 4 methods on {count} images...")
    
    methods = {
        "LSD (Proposed Latent Cosine)": pipeline.get("lsd_divergence", pipeline.get("lsd")),
        "LaRE² (CVPR 2024 Latent L2)": pipeline.get("baselines", {}).get("lare2", pipeline.get("lare2")),
        "DIRE (ICCV 2023 Pixel Diff)": pipeline.get("baselines", {}).get("dire", pipeline.get("dire")),
        "ADRD (PLOS ONE 2026 Discrepancy)": pipeline.get("baselines", {}).get("adrd", pipeline.get("adrd"))
    }
    
    results = {}
    for name, method in methods.items():
        print(f"Running {name}...")
        y_true, y_scores = [], []
        for i, batch in enumerate(loader):
            if i >= 10: break
            imgs = batch["image"].to(pipeline["device"], dtype=torch.float32)
            lbls = batch["label"].cpu().numpy()
            with torch.no_grad():
                out = method.compute_divergence(imgs)
                raw_msi = out["msi"]
                if isinstance(raw_msi, list):
                    scores = raw_msi
                elif hasattr(raw_msi, "cpu"):
                    scores = raw_msi.cpu().numpy().tolist()
                else:
                    scores = list(raw_msi)
            lbls_list = lbls.tolist() if hasattr(lbls, "tolist") else list(lbls)
            y_true.extend(lbls_list)
            y_scores.extend(scores)
        
        m = compute_detection_metrics(y_true, y_scores)
        results[name] = m
        ap_val = m.get("ap", m.get("average_precision", 0.0))
        auroc_val = m.get("auroc", 0.5)
        print(f"  -> {name} | AUROC: {auroc_val:.4f} | AP: {ap_val:.4f}")
        
    summary_path = f"{PROJECT_OUTPUT_DIR}/metrics/4way_baselines_summary.json"
    with open(summary_path, "w") as f:
        json.dump(results, f, indent=2)
    print(f"\n✅ 4-Way Baselines saved directly to Mac project folder: {summary_path}")
else:
    print("Target dataset not found. Please run Step 5 first.")

### 🟢 Step 10: Functionality 4 — Degradation Robustness Suite (Auto-Saved to Project Folder)
Tests robustness under real-world social media artifacts: JPEG compression (qualities 95, 75, 50, 30), Gaussian blur, and resizing.

In [ ]:
import os, json, torch, numpy as np
from lsd_core.data.dataset_loader import get_paper_dataloader
from lsd_core.algorithms.degradations import apply_jpeg_compression, apply_gaussian_blur, apply_resize_degradation

if "PROJECT_OUTPUT_DIR" not in globals():
    PROJECT_OUTPUT_DIR = "/content/drive/MyDrive/Research_Pdf_Outputs" if os.path.exists("/content/drive/MyDrive") else "/content/outputs"
if "target_dir" not in globals() or not target_dir:
    for cand in ["/content/data/140k_faces", "/content/data/paper_benchmark", "/content/data/faceforensics", "./data/paper_benchmark"]:
        if os.path.exists(cand) and len(os.listdir(cand)) > 0:
            target_dir = cand
            break

def _calc_mean_msi(out_dict):
    val = out_dict["msi"]
    if isinstance(val, (list, tuple)):
        return float(np.mean(val))
    elif hasattr(val, "mean"):
        return float(val.mean().item())
    return float(val)

if target_dir:
    loader, _ = get_paper_dataloader(target_dir, batch_size=2)
    sample_batch = next(iter(loader))["image"].to(pipeline["device"], dtype=torch.float32)
    
    lsd_eng = pipeline.get("lsd_divergence", pipeline.get("lsd"))
    with torch.no_grad():
        base_msi = _calc_mean_msi(lsd_eng.compute_divergence(sample_batch))
        
        jpeg_scores = {}
        for q in [95, 75, 50, 30]:
            deg_img = apply_jpeg_compression(sample_batch, quality=q)
            jpeg_scores[f"Q{q}"] = _calc_mean_msi(lsd_eng.compute_divergence(deg_img))
            
        blur_img = apply_gaussian_blur(sample_batch, kernel_size=5, sigma=1.0)
        blur_msi = _calc_mean_msi(lsd_eng.compute_divergence(blur_img))
        
        resize_img = apply_resize_degradation(sample_batch, scale_factor=0.5)
        resize_msi = _calc_mean_msi(lsd_eng.compute_divergence(resize_img))
        
    rob_results = {
        "clean_baseline_msi": base_msi,
        "jpeg_compression": jpeg_scores,
        "gaussian_blur_msi": blur_msi,
        "resize_0.5x_msi": resize_msi
    }
    rob_path = f"{PROJECT_OUTPUT_DIR}/metrics/robustness_summary.json"
    with open(rob_path, "w") as f:
        json.dump(rob_results, f, indent=2)
        
    print(f"Clean Baseline MSI: {base_msi:.4f}")
    print("JPEG Robustness:", {k: f"{v:.4f}" for k, v in jpeg_scores.items()})
    print(f"Gaussian Blur MSI: {blur_msi:.4f}")
    print(f"Resize 0.5x MSI: {resize_msi:.4f}")
    print(f"\n✅ Robustness summary saved directly to Mac project folder: {rob_path}")

### 🟢 Step 11: Functionality 5 — Interactive Single-Image Heatmap Inspector (Auto-Saved to Project Folder)
Inspect any deepfake or real image and visualize where the manipulation is localized.

In [ ]:
import glob, os
from PIL import Image
import matplotlib.pyplot as plt
from main import run_single_image_inspection

if "PROJECT_OUTPUT_DIR" not in globals():
    PROJECT_OUTPUT_DIR = "/content/drive/MyDrive/Research_Pdf_Outputs" if os.path.exists("/content/drive/MyDrive") else "/content/outputs"

all_samples = glob.glob("/content/data/**/*.jpg", recursive=True) + glob.glob("/content/data/**/*.png", recursive=True)
if not all_samples:
    all_samples = glob.glob("./data/**/*.jpg", recursive=True) + glob.glob("./data/**/*.png", recursive=True)

if all_samples:
    test_img = all_samples[0]
    print(f"Inspecting sample: {test_img}")
    out_inspect = f"{PROJECT_OUTPUT_DIR}/inspection.png"
    res = run_single_image_inspection(pipeline, test_img, output_path=out_inspect)
    if isinstance(res, dict):
        pred = res.get("predicted_label", "Evaluated")
        score = res.get("msi_score", 0.0)
    else:
        pred = "Evaluated"
        score = 0.0
    print(f"\nPrediction: {pred} (MSI Score: {score:.4f})")
    
    if os.path.exists(out_inspect):
        plt.figure(figsize=(10, 4))
        plt.imshow(Image.open(out_inspect))
        plt.axis("off")
        plt.show()
    print(f"\n✅ Inspection heatmap saved directly to Mac project folder: {out_inspect}")
else:
    print("No sample image found for inspection.")

### 🟢 Step 12: 1-Click Backup Download to Mac Downloads Folder
In addition to live Google Drive syncing, run this cell anytime to download all outputs as a `.zip` file.

In [ ]:
from google.colab import files
import shutil, os

zip_archive_path = "/content/Research_Pdf_Outputs"
source_dir = PROJECT_OUTPUT_DIR if os.path.exists(PROJECT_OUTPUT_DIR) else "/content/outputs"
shutil.make_archive(zip_archive_path, "zip", source_dir)
files.download(f"{zip_archive_path}.zip")
print(f"\n✅ Outputs archive downloading to your Mac: {zip_archive_path}.zip")